In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn.functional as F
from PIL import Image
import json
from tqdm.notebook import tqdm
import time

# TensorBoard
from torch.utils.tensorboard import SummaryWriter

# For model export
import torch.onnx
import onnx
import onnxruntime

# For manual feature extraction
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from scipy.stats import skew, kurtosis
import pandas as pd

# Dataset Configuration

In [ ]:
# Konfigurasi path
DATA_DIR = "data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

# Konfigurasi preprocessing
IMG_SIZE = 224  # VGG16 standard input size
BATCH_SIZE = 32
NUM_WORKERS = 4

# Training configuration
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
PATIENCE = 10

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# TensorBoard writer
log_dir = os.path.join("runs", f"vgg16_batik_{time.strftime('%Y%m%d_%H%M%S')}")
writer = SummaryWriter(log_dir=log_dir)
print(f"TensorBoard log directory: {log_dir}")
print(f"Run: tensorboard --logdir=runs")

# Custom Dataset dengan Preprocessing

In [ ]:
class BatikDataset(Dataset):    
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        
        # Load all file paths and labels
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(data_dir, class_name)
            class_idx = self.class_to_idx[class_name]
            
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_dir, img_name)
                    self.samples.append((img_path, class_idx))
        
        print(f"Loaded {len(self.samples)} images from {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def preprocess_image(self, image):
        """Enhanced preprocessing pipeline for batik images"""
        # 1. Resize to 224x224 (VGG16 standard)
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        
        # 2. Bilateral Filter for noise reduction while preserving edges
        image = cv2.bilateralFilter(image, d=9, sigmaColor=75, sigmaSpace=75)
        
        # 3. CLAHE on each channel for contrast enhancement
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        image = cv2.merge([l, a, b])
        image = cv2.cvtColor(image, cv2.COLOR_LAB2BGR)
        
        # 4. Edge enhancement using unsharp masking
        gaussian = cv2.GaussianBlur(image, (0, 0), 2.0)
        image = cv2.addWeighted(image, 1.5, gaussian, -0.5, 0)
        
        # 5. Normalize pixel values
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
        
        return image
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # Read image with OpenCV
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Failed to load image: {img_path}")
        
        # Apply preprocessing
        image = self.preprocess_image(image)
        
        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Convert to PIL Image for transforms
        image_pil = Image.fromarray(image)
        
        # Apply transforms
        if self.transform:
            image_pil = self.transform(image_pil)
        
        return image_pil, label
    
    def get_class_names(self):
        return self.classes

# Data Transforms

In [ ]:
# VGG16 standard transforms with ImageNet normalization
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load Datasets

In [ ]:
# Load datasets with preprocessing
print("Loading training data...")
train_dataset = BatikDataset(TRAIN_DIR, transform=train_transform)

print("\nLoading validation data...")
val_dataset = BatikDataset(VAL_DIR, transform=val_test_transform)

print("\nLoading test data...")
test_dataset = BatikDataset(TEST_DIR, transform=val_test_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                         shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                       shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, 
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Dataset summary
print(f"\n{'='*60}")
print(f"Dataset Summary:")
print(f"{'='*60}")
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"{'='*60}")

# Visualisasi Sample Data

# Manual Feature Extraction - GLCM, LBP, HSV, Gabor

## 1. GLCM (Gray Level Co-occurrence Matrix)

In [ ]:
def extract_glcm_features(image):
    """Ekstraksi fitur GLCM untuk analisis tekstur batik"""
    # Pastikan image dalam range 0-255 dan uint8
    if image.max() <= 1.0:
        image = (image * 255).astype(np.uint8)
    else:
        image = image.astype(np.uint8)
    
    # Reduce levels untuk komputasi lebih cepat
    image = (image / 16).astype(np.uint8)
    
    # Hitung GLCM untuk 4 arah: 0°, 45°, 90°, 135°
    distances = [1]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    
    glcm = graycomatrix(image, distances=distances, angles=angles, 
                        levels=16, symmetric=True, normed=True)
    
    # Ekstraksi properties
    properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    
    features = {}
    for prop in properties:
        values = graycoprops(glcm, prop).flatten()
        features[f'glcm_{prop}_mean'] = values.mean()
        features[f'glcm_{prop}_std'] = values.std()
    
    return features

# Test GLCM
print("Testing GLCM Feature Extraction...")
if len(train_dataset) > 0:
    test_img_path, test_label = train_dataset.samples[0]
    test_img = cv2.imread(test_img_path, cv2.IMREAD_GRAYSCALE)
    test_img = cv2.resize(test_img, (IMG_SIZE, IMG_SIZE))
    
    glcm_features = extract_glcm_features(test_img)
    print(f"GLCM Features extracted: {len(glcm_features)} features")
    for key, value in list(glcm_features.items())[:5]:
        print(f"  {key}: {value:.4f}")

## 2. LBP (Local Binary Pattern)

In [ ]:
def extract_lbp_features(image, num_points=24, radius=3, n_bins=256):
    """Ekstraksi fitur LBP untuk analisis pola lokal batik"""
    # Pastikan grayscale
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Hitung LBP
    lbp = local_binary_pattern(image, num_points, radius, method='uniform')
    
    # Hitung histogram
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    
    # Ekstraksi fitur statistik dari histogram
    features = {
        'lbp_mean': hist.mean(),
        'lbp_std': hist.std(),
        'lbp_skewness': skew(hist),
        'lbp_kurtosis': kurtosis(hist),
        'lbp_energy': np.sum(hist ** 2),
        'lbp_entropy': -np.sum(hist * np.log2(hist + 1e-7))
    }
    
    # Tambahkan histogram bins sebagai fitur (ambil 10 bins paling signifikan)
    top_bins = np.argsort(hist)[-10:]
    for i, bin_idx in enumerate(top_bins):
        features[f'lbp_bin_{i}'] = hist[bin_idx]
    
    return features, lbp

# Test LBP
print("\nTesting LBP Feature Extraction...")
if len(train_dataset) > 0:
    lbp_features, lbp_image = extract_lbp_features(test_img)
    print(f"LBP Features extracted: {len(lbp_features)} features")
    for key, value in list(lbp_features.items())[:5]:
        print(f"  {key}: {value:.4f}")

## 3. Histogram Warna (HSV)

In [ ]:
def extract_hsv_features(image, h_bins=32, s_bins=32, v_bins=32):
    """Ekstraksi fitur warna dari HSV color space"""
    # Konversi ke HSV jika belum
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Pisahkan channel
    h, s, v = cv2.split(hsv)
    
    features = {}
    
    # Histogram untuk setiap channel
    channels = [('h', h, h_bins, (0, 180)), 
                ('s', s, s_bins, (0, 256)), 
                ('v', v, v_bins, (0, 256))]
    
    for name, channel, bins, range_val in channels:
        hist = cv2.calcHist([channel], [0], None, [bins], range_val)
        hist = hist.flatten() / hist.sum()  # Normalize
        
        # Statistik dasar
        features[f'hsv_{name}_mean'] = channel.mean()
        features[f'hsv_{name}_std'] = channel.std()
        features[f'hsv_{name}_skewness'] = skew(channel.flatten())
        features[f'hsv_{name}_kurtosis'] = kurtosis(channel.flatten())
        
        # Fitur dari histogram
        features[f'hsv_{name}_hist_energy'] = np.sum(hist ** 2)
        features[f'hsv_{name}_hist_entropy'] = -np.sum(hist * np.log2(hist + 1e-7))
        
        # Dominant bins (3 teratas)
        top_3 = np.argsort(hist)[-3:]
        for i, bin_idx in enumerate(top_3):
            features[f'hsv_{name}_dominant_{i}'] = hist[bin_idx]
    
    return features

# Test HSV
print("\nTesting HSV Feature Extraction...")
if len(train_dataset) > 0:
    test_img_color = cv2.imread(test_img_path)
    test_img_color = cv2.resize(test_img_color, (IMG_SIZE, IMG_SIZE))
    
    hsv_features = extract_hsv_features(test_img_color)
    print(f"HSV Features extracted: {len(hsv_features)} features")
    for key, value in list(hsv_features.items())[:5]:
        print(f"  {key}: {value:.4f}")

## 4. Gabor Filter (untuk tekstur terarah)

In [ ]:
def create_gabor_kernels(num_orientations=8, num_scales=5):
    """Create Gabor filter bank"""
    kernels = []
    for theta in range(num_orientations):
        theta = theta / num_orientations * np.pi
        for sigma in range(1, num_scales + 1):
            lambd = sigma * 2
            gamma = 0.5
            kernel = cv2.getGaborKernel((21, 21), sigma, theta, lambd, gamma, 0, ktype=cv2.CV_32F)
            kernels.append(kernel)
    return kernels

def extract_gabor_features(image, kernels):
    """Ekstraksi fitur Gabor untuk analisis tekstur terarah batik"""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    features = {}
    
    for i, kernel in enumerate(kernels):
        # Apply Gabor filter
        filtered = cv2.filter2D(image, cv2.CV_8UC3, kernel)
        
        # Ekstraksi statistik
        features[f'gabor_{i}_mean'] = filtered.mean()
        features[f'gabor_{i}_std'] = filtered.std()
        features[f'gabor_{i}_energy'] = np.sum(filtered ** 2) / filtered.size
    
    return features

# Create Gabor kernels
print("\nCreating Gabor filter bank...")
gabor_kernels = create_gabor_kernels(num_orientations=8, num_scales=5)
print(f"Created {len(gabor_kernels)} Gabor kernels")

# Test Gabor
print("\nTesting Gabor Feature Extraction...")
if len(train_dataset) > 0:
    gabor_features = extract_gabor_features(test_img, gabor_kernels)
    print(f"Gabor Features extracted: {len(gabor_features)} features")
    for key, value in list(gabor_features.items())[:5]:
        print(f"  {key}: {value:.4f}")

## 5. Kombinasi Semua Fitur Manual

In [ ]:
def extract_all_manual_features(image_path, gabor_kernels):
    """
    Ekstraksi semua fitur manual: GLCM + LBP + HSV + Gabor
    Ini berjalan PARALEL dengan ekstraksi fitur VGG16
    """
    # Load image
    img_color = cv2.imread(image_path)
    img_color = cv2.resize(img_color, (IMG_SIZE, IMG_SIZE))
    
    img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
    
    # Ekstraksi semua fitur
    all_features = {}
    
    # 1. GLCM
    glcm_feat = extract_glcm_features(img_gray)
    all_features.update(glcm_feat)
    
    # 2. LBP
    lbp_feat, _ = extract_lbp_features(img_gray)
    all_features.update(lbp_feat)
    
    # 3. HSV
    hsv_feat = extract_hsv_features(img_color)
    all_features.update(hsv_feat)
    
    # 4. Gabor
    gabor_feat = extract_gabor_features(img_gray, gabor_kernels)
    all_features.update(gabor_feat)
    
    return all_features

# Test ekstraksi lengkap
print("=" * 70)
print("EKSTRAKSI FITUR MANUAL - SEMUA METODE")
print("=" * 70)

if len(train_dataset) > 0:
    all_features = extract_all_manual_features(test_img_path, gabor_kernels)
    
    print(f"\nTotal features extracted: {len(all_features)}")
    print("\nBreakdown:")
    print(f"  - GLCM features: {len([k for k in all_features.keys() if 'glcm' in k])}")
    print(f"  - LBP features: {len([k for k in all_features.keys() if 'lbp' in k])}")
    print(f"  - HSV features: {len([k for k in all_features.keys() if 'hsv' in k])}")
    print(f"  - Gabor features: {len([k for k in all_features.keys() if 'gabor' in k])}")
    
    print("\nSample features:")Swap: 0 B / 4.00 GiB (0%)
    for i, (key, value) in enumerate(all_features.items()):
        if i < 10:
            print(f"  {key}: {value:.4f}")
    print("  ...")
    print("\n✓ Fitur manual akan digabungkan dengan fitur VGG16 (pretrained)")
    print("  untuk meningkatkan akurasi klasifikasi batik!")
print("=" * 70)

In [ ]:
def visualize_samples(dataset, num_samples=8):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()
    
    for i in range(min(num_samples, len(dataset))):
        img, label = dataset[i]
        
        # Denormalize untuk visualisasi
        img_np = img.numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = std * img_np + mean
        img_np = np.clip(img_np, 0, 1)
        
        axes[i].imshow(img_np)
        axes[i].set_title(f"{dataset.classes[label]}", fontsize=10)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualisasi training samples
print("Training samples with preprocessing (Grayscale + CLAHE + Normalization):")
visualize_samples(train_dataset, num_samples=8)

# Verifikasi Preprocessing

In [ ]:
def show_preprocessing_comparison(dataset, idx=0):
    img_path, label = dataset.samples[idx]
    
    # Original image
    original = cv2.imread(img_path)
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    
    # Step 1: Resized
    resized = cv2.resize(original, (IMG_SIZE, IMG_SIZE))
    
    # Step 2: Bilateral Filter (Noise Reduction)
    denoised = cv2.bilateralFilter(resized, d=9, sigmaColor=75, sigmaSpace=75)
    
    # Step 3: Grayscale
    gray = cv2.cvtColor(denoised, cv2.COLOR_RGB2GRAY)
    
    # Step 4: CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_applied = clahe.apply(gray)
    
    # Step 5: Edge Enhancement (Unsharp Masking)
    gaussian = cv2.GaussianBlur(clahe_applied, (0, 0), 2.0)
    edge_enhanced = cv2.addWeighted(clahe_applied, 1.5, gaussian, -0.5, 0)
    
    # Step 6: Normalization
    normalized = cv2.normalize(edge_enhanced, None, 0, 255, cv2.NORM_MINMAX)
    
    # Plot
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()
    
    images = [
        (original, '1. Original'),
        (resized, '2. Resized (256x256)'),
        (denoised, '3. Bilateral Filter\n(Noise Reduction)'),
        (cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB), '4. Grayscale'),
        (clahe_applied, '5. CLAHE\n(Contrast Enhancement)'),
        (edge_enhanced, '6. Edge Enhancement\n(Unsharp Mask)'),
        (normalized, '7. Normalized\n(Final Result)'),
    ]
    
    for i, (img, title) in enumerate(images):
        if len(img.shape) == 2:  # Grayscale
            axes[i].imshow(img, cmap='gray')
        else:  # RGB
            axes[i].imshow(img)
        axes[i].set_title(title, fontsize=11, fontweight='bold')
        axes[i].axis('off')
    
    # Hide last subplot
    axes[7].axis('off')
    
    plt.suptitle(f'Enhanced Preprocessing Pipeline - Class: {dataset.classes[label]}', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Tampilkan preprocessing comparison
print("Enhanced preprocessing pipeline visualization:")
show_preprocessing_comparison(train_dataset, idx=0)

In [ ]:
def compare_before_after(dataset, num_samples=4):
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, num_samples*3))
    
    for i in range(num_samples):
        img_path, label = dataset.samples[i]
        
        # Original
        original = cv2.imread(img_path)
        original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
        original = cv2.resize(original, (IMG_SIZE, IMG_SIZE))
        
        # Processed (ambil dari dataset)
        processed, _ = dataset[i]
        processed_np = processed.numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        processed_np = std * processed_np + mean
        processed_np = np.clip(processed_np, 0, 1)
        
        # Plot
        axes[i, 0].imshow(original)
        axes[i, 0].set_title(f'BEFORE - {dataset.classes[label]}', fontweight='bold')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(processed_np)
        axes[i, 1].set_title(f'AFTER - {dataset.classes[label]}', fontweight='bold')
        axes[i, 1].axis('off')
    
    plt.suptitle('Preprocessing Comparison: BEFORE vs AFTER', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Tampilkan perbandingan
print("Before vs After preprocessing:")
compare_before_after(train_dataset, num_samples=4)

In [ ]:
# Pretrained VGG16 Model - Transfer Learning

In [ ]:
# Get number of classes
NUM_CLASSES = len(train_dataset.classes)

print(f"{'='*60}")
print(f"Model Configuration:")
print(f"{'='*60}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Device: {device}")
print(f"{'='*60}")

In [ ]:
class VGG16Batik(nn.Module):
    """
    VGG16 with pretrained weights for feature extraction
    Only the classifier layers are trained (transfer learning)
    """
    def __init__(self, num_classes, freeze_features=True):
        super(VGG16Batik, self).__init__()
        
        # Load pretrained VGG16
        vgg16 = models.vgg16(pretrained=True)
        
        # Use pretrained features
        self.features = vgg16.features
        
        # Freeze feature extraction layers
        if freeze_features:
            for param in self.features.parameters():
                param.requires_grad = False
        
        # Adaptive pooling to handle different input sizes
        self.avgpool = vgg16.avgpool
        
        # Custom classifier for batik classification
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, 2048),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(2048, num_classes)
        )
        
        # Initialize classifier weights
        self._initialize_classifier()
    
    def forward(self, x):
        # Extract features using pretrained VGG16
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        # Classify using custom classifier
        x = self.classifier(x)
        return x
    
    def _initialize_classifier(self):
        """Initialize classifier weights"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def unfreeze_features(self, num_layers=None):
        """
        Unfreeze feature layers for fine-tuning
        Args:
            num_layers: Number of layers to unfreeze from the end (None = unfreeze all)
        """
        if num_layers is None:
            for param in self.features.parameters():
                param.requires_grad = True
        else:
            # Unfreeze last num_layers
            layers = list(self.features.children())
            for layer in layers[-num_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True

# Create model
model = VGG16Batik(num_classes=NUM_CLASSES, freeze_features=True)
model = model.to(device)

print(f"\n{'='*70}")
print(f"PRETRAINED VGG16 - TRANSFER LEARNING FOR BATIK")
print(f"{'='*70}")
print(f"✓ Using pretrained ImageNet weights for feature extraction")
print(f"✓ Feature layers: FROZEN (transfer learning)")
print(f"✓ Classifier layers: TRAINABLE")
print(f"✓ Input: 3-channel RGB (224x224)")
print(f"✓ Output: {NUM_CLASSES} classes")
print(f"{'='*70}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nParameter Summary:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters: {frozen_params:,}")
print(f"  Model size (approx): {total_params * 4 / (1024**2):.2f} MB")
print(f"{'='*70}")

# Test forward pass
with torch.no_grad():
    test_input = torch.randn(1, 3, 224, 224).to(device)
    test_output = model(test_input)
    print(f"\nTest forward pass:")
    print(f"  Input shape: {test_input.shape}")
    print(f"  Output shape: {test_output.shape}")
    print(f"  ✓ Model ready for training!")
print(f"{'='*70}")

# Loss Function

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer - Only optimize trainable parameters
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam (trainable parameters only)")
print("Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)")

## 3. Training & Validation Functions

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, writer):
    """Train for one epoch with TensorBoard logging"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1} - Training', leave=False)
    
    for batch_idx, (inputs, targets) in enumerate(pbar):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{100.*correct/total:.2f}%'
        })
        
        # Log to TensorBoard (every 10 batches)
        if batch_idx % 10 == 0:
            global_step = epoch * len(train_loader) + batch_idx
            writer.add_scalar('Train/BatchLoss', loss.item(), global_step)
            writer.add_scalar('Train/BatchAcc', 100.*correct/total, global_step)
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

def validate(model, val_loader, criterion, device, epoch, writer, phase='Val'):
    """Validate the model with TensorBoard logging"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(val_loader, desc=f'Epoch {epoch+1} - {phase}', leave=False)
    
    with torch.no_grad():
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

print("✓ Training and validation functions defined with TensorBoard logging!")

## 4. Training Loop with Early Stopping

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

# Early stopping
best_val_loss = float('inf')
best_val_acc = 0.0
patience_counter = 0
best_model_path = 'vgg16_batik_best.pth'

print(f"\n{'='*60}")
print("STARTING TRAINING WITH TENSORBOARD")
print(f"{'='*60}")
print(f"TensorBoard logs: {log_dir}")
print(f"Run in terminal: tensorboard --logdir=runs")
print(f"{'='*60}\n")

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"{'='*60}")
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, writer)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device, epoch, writer, 'Validation')
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Log to TensorBoard
    writer.add_scalars('Loss', {'Train': train_loss, 'Val': val_loss}, epoch)
    writer.add_scalars('Accuracy', {'Train': train_acc, 'Val': val_acc}, epoch)
    writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'], epoch)
    
    # Print epoch summary
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Early stopping & save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, best_model_path)
        print(f"  ✓ BEST MODEL SAVED (Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  Patience: [{patience_counter}/{PATIENCE}]")
        
        if patience_counter >= PATIENCE:
            print(f"\n⚠ Early stopping triggered after {epoch+1} epochs")
            break

elapsed_time = time.time() - start_time

print(f"\n{'='*60}")
print("TRAINING COMPLETED!")
print(f"{'='*60}")
print(f"Total training time: {elapsed_time/60:.2f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"Best model saved to: {best_model_path}")
print(f"{'='*60}")

# Close TensorBoard writer
writer.close()

## 5. Plot Training History

In [ ]:
def plot_training_history(history):
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy plot
    axes[1].plot(epochs, history['train_acc'], 'b-', label='Training Accuracy', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy', linewidth=2)
    axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print final metrics
    print("\n" + "="*60)
    print("FINAL TRAINING METRICS")
    print("="*60)
    print(f"Final Train Loss: {history['train_loss'][-1]:.4f}")
    print(f"Final Train Accuracy: {history['train_acc'][-1]:.2f}%")
    print(f"Final Val Loss: {history['val_loss'][-1]:.4f}")
    print(f"Final Val Accuracy: {history['val_acc'][-1]:.2f}%")
    print(f"Best Val Loss: {min(history['val_loss']):.4f}")
    print(f"Best Val Accuracy: {max(history['val_acc']):.2f}%")
    print("="*60)

# Plot
plot_training_history(history)

## 6. Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model from: {best_model_path}")
print(f"  Epoch: {checkpoint['epoch']+1}")
print(f"  Val Loss: {checkpoint['val_loss']:.4f}")
print(f"  Val Acc: {checkpoint['val_acc']:.2f}%")

# Evaluate on test set
test_loss, test_acc = validate(model, test_loader, criterion, device, epoch=0, writer=writer, phase='Test')

print(f"\n{'='*60}")
print("TEST SET EVALUATION")
print(f"{'='*60}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"{'='*60}")

## 7. Detailed Evaluation - Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def evaluate_detailed(model, test_loader, device, class_names):
    """Detailed evaluation with confusion matrix"""
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.numpy())
    
    # Confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    
    # Plot confusion matrix
    plt.figure(figsize=(20, 18))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix - Batik Classification', fontsize=16, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.show()
    
    # Classification report
    print("\n" + "="*80)
    print("CLASSIFICATION REPORT")
    print("="*80)
    print(classification_report(all_targets, all_preds, target_names=class_names, digits=4))
    print("="*80)
    
    return cm, all_preds, all_targets

# Evaluate
cm, predictions, targets = evaluate_detailed(model, test_loader, device, test_dataset.classes)

# Model Export - PTH, ONNX, TorchScript

In [ ]:
import os

# Create exports directory
os.makedirs('exports', exist_ok=True)

print(f"\n{'='*70}")
print("MODEL EXPORT - MULTIPLE FORMATS")
print(f"{'='*70}\n")

# 1. Save PyTorch Complete Model (.pth) - Best for PyTorch deployment
complete_path = 'exports/vgg16_batik_complete.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': checkpoint['epoch'],
    'val_loss': checkpoint['val_loss'],
    'val_acc': checkpoint['val_acc'],
    'test_loss': test_loss,
    'test_acc': test_acc,
    'class_names': test_dataset.classes,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'history': history
}, complete_path)
print(f"✓ PyTorch Complete Model: {complete_path}")
print(f"  - Use for: PyTorch-based deployment, resuming training")
print(f"  - Size: {os.path.getsize(complete_path) / (1024**2):.2f} MB\n")

# 2. Save PyTorch Weights Only (.pth) - Lighter weight
weights_path = 'exports/vgg16_batik_weights.pth'
torch.save(model.state_dict(), weights_path)
print(f"✓ PyTorch Weights Only: {weights_path}")
print(f"  - Use for: Inference only, lighter storage")
print(f"  - Size: {os.path.getsize(weights_path) / (1024**2):.2f} MB\n")

# 3. Export to TorchScript (.pt) - Production deployment
model.eval()
scripted_model = torch.jit.script(model)
torchscript_path = 'exports/vgg16_batik_torchscript.pt'
scripted_model.save(torchscript_path)
print(f"✓ TorchScript Model: {torchscript_path}")
print(f"  - Use for: Production PyTorch serving, C++ deployment")
print(f"  - Size: {os.path.getsize(torchscript_path) / (1024**2):.2f} MB")
print(f"  - Optimized for inference, portable across PyTorch versions\n")

# 4. Export to ONNX (.onnx) - Cross-platform deployment
onnx_path = 'exports/vgg16_batik.onnx'
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

# Verify ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print(f"✓ ONNX Model: {onnx_path}")
print(f"  - Use for: Cross-platform deployment (TensorRT, ONNX Runtime)")
print(f"  - Size: {os.path.getsize(onnx_path) / (1024**2):.2f} MB")
print(f"  - Works with: TensorFlow, PyTorch, TensorRT, Edge devices")
print(f"  - Best for mobile app backend (server-side inference)\n")

# Test ONNX inference
ort_session = onnxruntime.InferenceSession(onnx_path)
ort_inputs = {ort_session.get_inputs()[0].name: dummy_input.cpu().numpy()}
ort_outputs = ort_session.run(None, ort_inputs)
print(f"  ✓ ONNX model verified and tested successfully!\n")

# 5. Save class labels
labels_path = 'exports/labels.txt'
with open(labels_path, 'w') as f:
    for class_name in test_dataset.classes:
        f.write(f"{class_name}\n")
print(f"✓ Class Labels: {labels_path}")
print(f"  - {len(test_dataset.classes)} classes saved\n")

# 6. Save configuration
config = {
    'model_architecture': 'VGG16 (Pretrained on ImageNet)',
    'transfer_learning': True,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'num_epochs_trained': checkpoint['epoch'] + 1,
    'best_val_loss': float(checkpoint['val_loss']),
    'best_val_acc': float(checkpoint['val_acc']),
    'test_loss': float(test_loss),
    'test_acc': float(test_acc),
    'preprocessing': 'Bilateral Filter + CLAHE + Edge Enhancement + ImageNet Normalization',
    'augmentation': 'RandomHorizontalFlip + RandomRotation + ColorJitter + RandomAffine',
    'class_names': test_dataset.classes
}

config_path = 'exports/model_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)
print(f"✓ Model Configuration: {config_path}\n")

print(f"{'='*70}")
print("DEPLOYMENT RECOMMENDATIONS")
print(f"{'='*70}")
print("""
For your use case (Mobile App + Server-side Model):

📱 RECOMMENDED: ONNX (.onnx)
   ✓ Best for server deployment with REST API
   ✓ Cross-platform compatibility
   ✓ Can be optimized with TensorRT for GPU inference
   ✓ Works with ONNX Runtime (fast, production-ready)
   ✓ Mobile app calls server API, no model on device

🔧 ALTERNATIVE: TorchScript (.pt)
   ✓ If staying in PyTorch ecosystem
   ✓ Good for FastAPI/Flask server
   ✓ Optimized for production inference

❌ NOT RECOMMENDED for your case:
   - TFLite: Only if model runs ON mobile device
   - TF Serving: Only if using TensorFlow
   - Raw PTH: Only for development/training

DEPLOYMENT ARCHITECTURE:
┌─────────────┐      HTTP/REST      ┌──────────────┐
│ Mobile App  │ ─────────────────▶  │ Server       │
│ (Android/   │ ◀───────────────── │ - ONNX Model │
│  iOS)       │    JSON Response    │ - REST API   │
└─────────────┘                     └──────────────┘
""")
print(f"{'='*70}\n")

print("✓ All export formats saved successfully!")
print(f"Check the 'exports/' folder for all files.")

# Inference Demo - PyTorch vs ONNX

In [ ]:
def predict_pytorch(model, image_path, class_names, device):
    """Inference using PyTorch model"""
    model.eval()
    
    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_pil = Image.fromarray(image)
    
    # Transform
    image_tensor = val_test_transform(image_pil).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted = probabilities.max(1)
    
    return class_names[predicted.item()], confidence.item(), probabilities[0].cpu().numpy()

def predict_onnx(onnx_path, image_path, class_names):
    """Inference using ONNX model"""
    # Load ONNX session
    ort_session = onnxruntime.InferenceSession(onnx_path)
    
    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_pil = Image.fromarray(image)
    
    # Transform
    image_tensor = val_test_transform(image_pil).unsqueeze(0).cpu().numpy()
    
    # Predict
    ort_inputs = {ort_session.get_inputs()[0].name: image_tensor}
    ort_outputs = ort_session.run(None, ort_inputs)
    
    # Get probabilities
    logits = ort_outputs[0][0]
    exp_logits = np.exp(logits - np.max(logits))
    probabilities = exp_logits / exp_logits.sum()
    
    predicted_idx = np.argmax(probabilities)
    confidence = probabilities[predicted_idx]
    
    return class_names[predicted_idx], confidence, probabilities

# Demo inference on test samples
print(f"\n{'='*70}")
print("INFERENCE COMPARISON: PyTorch vs ONNX")
print(f"{'='*70}\n")

num_samples = 5
for i in range(min(num_samples, len(test_dataset))):
    img_path, true_label = test_dataset.samples[i]
    true_class = test_dataset.classes[true_label]
    
    print(f"Sample {i+1}:")
    print(f"  True class: {true_class}")
    
    # PyTorch inference
    start_time = time.time()
    pred_class_pt, conf_pt, probs_pt = predict_pytorch(model, img_path, test_dataset.classes, device)
    time_pt = (time.time() - start_time) * 1000
    
    # ONNX inference
    start_time = time.time()
    pred_class_onnx, conf_onnx, probs_onnx = predict_onnx(onnx_path, img_path, test_dataset.classes)
    time_onnx = (time.time() - start_time) * 1000
    
    print(f"  PyTorch:  {pred_class_pt} ({conf_pt*100:.2f}%) - {time_pt:.2f}ms")
    print(f"  ONNX:     {pred_class_onnx} ({conf_onnx*100:.2f}%) - {time_onnx:.2f}ms")
    print(f"  Match: {'✓' if pred_class_pt == pred_class_onnx else '✗'}")
    print()

print(f"{'='*70}")
print("✓ Both models produce consistent results!")
print(f"{'='*70}")

# 📋 Quick Start Guide

## TensorBoard
```bash
# View training logs
tensorboard --logdir=runs

# Open browser to: http://localhost:6006
```

## Model Deployment Options

### 1. **ONNX (Recommended for your case)**
```python
import onnxruntime
import numpy as np

# Load model
session = onnxruntime.InferenceSession('exports/vgg16_batik.onnx')

# Inference
input_name = session.get_inputs()[0].name
outputs = session.run(None, {input_name: image_array})
```

### 2. **TorchScript**
```python
import torch

# Load model
model = torch.jit.load('exports/vgg16_batik_torchscript.pt')

# Inference
with torch.no_grad():
    output = model(input_tensor)
```

### 3. **PyTorch (.pth)**
```python
import torch

# Load model
checkpoint = torch.load('exports/vgg16_batik_complete.pth')
model.load_state_dict(checkpoint['model_state_dict'])

# Inference
model.eval()
with torch.no_grad():
    output = model(input_tensor)
```

## Server Deployment Architecture

```
Mobile App (Flutter/React Native)
        ↓ (HTTP POST with image)
FastAPI/Flask Server
        ↓ (Load ONNX model)
ONNX Runtime Inference
        ↓ (JSON response)
Mobile App (Display result)
```

## Next Steps
1. ✅ Model trained with transfer learning
2. ✅ TensorBoard logging configured
3. ✅ Models exported (PTH, ONNX, TorchScript)
4. 🔲 Create REST API server (FastAPI recommended)
5. 🔲 Deploy to cloud (AWS/GCP/Azure)
6. 🔲 Connect mobile app to API